# 🧪 A/B Test Analysis — Statistical Significance & Impact Measurement

> **Dataset:** E-Commerce A/B Testing (Kaggle)  
> **Source:** https://www.kaggle.com/datasets/zhangluyuan/ab-testing  
> **Goal:** Evaluate whether a new landing page design drives a statistically significant improvement in conversion rate, calculate the true impact in revenue terms, and provide a clear go/no-go recommendation.

---

## Business Question

> *"Our product team shipped a new landing page. Did it actually improve conversions — or is the difference just random noise?"*

### Why This Matters

Without statistical rigour, teams make one of two expensive mistakes:
- **False positive** — ship a change that doesn't work, wasting engineering and design effort
- **False negative** — reject a change that does work, leaving revenue on the table

A/B testing with proper statistical testing eliminates both.

### Testing Framework

| Step | Method |
|---|---|
| Validity checks | Sample ratio mismatch, novelty effect |
| Primary metric | Conversion rate |
| Statistical test | Two-proportion Z-test |
| Significance level | α = 0.05 (95% confidence) |
| Power | β = 0.80 (80% power) |
| Decision | Lift, confidence interval, p-value, sample size check |

---
## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import scipy.stats as stats
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize
from statsmodels.stats.power import NormalIndPower
import warnings
warnings.filterwarnings('ignore')

# ── Palette ─────────────────────────────────────────────────
CHOCOLATE  = '#3d2314'
BROWN      = '#7a4f35'
CAMEL      = '#b89a74'
TERRACOTTA = '#c4694f'
SAND       = '#d9cdb8'
PARCHMENT  = '#ede5d4'
GREEN_OK   = '#5a8a5e'

plt.rcParams.update({
    'figure.facecolor':  PARCHMENT,
    'axes.facecolor':    '#f5f0e8',
    'axes.edgecolor':    SAND,
    'axes.labelcolor':   BROWN,
    'axes.titlecolor':   CHOCOLATE,
    'axes.titlesize':    13,
    'axes.titleweight':  'normal',
    'axes.labelsize':    10,
    'xtick.color':       BROWN,
    'ytick.color':       BROWN,
    'grid.color':        SAND,
    'grid.linestyle':    '--',
    'grid.alpha':        0.5,
    'font.family':       'serif',
    'text.color':        CHOCOLATE,
})

import os
os.makedirs('outputs', exist_ok=True)
print('Libraries loaded ✅')

---
## 2. Load & Inspect Data

Download from Kaggle: https://www.kaggle.com/datasets/zhangluyuan/ab-testing  
Save as `ab_data.csv` in `data/`.

In [ ]:
df = pd.read_csv('data/ab_data.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nGroup counts:')
print(df['group'].value_counts())
print(f'\nConversion counts:')
print(df['converted'].value_counts())
df.head(10)

---
## 3. Data Quality & Validity Checks

Before any statistical test, we must verify the experiment was run correctly. A flawed experiment cannot be rescued by statistics.

In [ ]:
print('═══ VALIDITY CHECKS ════════════════════════════════')

# ── Check 1: Sample Ratio Mismatch (SRM) ─────────────────────
control   = df[df['group'] == 'control']
treatment = df[df['group'] == 'treatment']
total = len(df)
ctrl_pct  = len(control) / total
treat_pct = len(treatment) / total

print(f'\n1. Sample Ratio Mismatch (SRM)')
print(f'   Control:   {len(control):,} ({ctrl_pct:.1%})')
print(f'   Treatment: {len(treatment):,} ({treat_pct:.1%})')
if abs(ctrl_pct - 0.5) < 0.02:
    print(f'   ✅ Split is approximately 50/50 — no SRM detected')
else:
    print(f'   ⚠️  SRM detected — split deviates from 50/50. Investigate before proceeding.')

# ── Check 2: Mismatched page assignments ─────────────────────
if 'landing_page' in df.columns:
    mismatch = df[
        ((df['group']=='control') & (df['landing_page']=='new_page')) |
        ((df['group']=='treatment') & (df['landing_page']=='old_page'))
    ]
    print(f'\n2. Mismatched Page Assignments')
    print(f'   Mismatched rows: {len(mismatch):,}')
    if len(mismatch) > 0:
        print(f'   ⚠️  Removing {len(mismatch):,} mismatched rows before analysis')
        df = df.drop(mismatch.index)
        control   = df[df['group'] == 'control']
        treatment = df[df['group'] == 'treatment']
    else:
        print('   ✅ No mismatches found')

# ── Check 3: Duplicate user IDs ──────────────────────────────
if 'user_id' in df.columns:
    dupes = df['user_id'].duplicated().sum()
    print(f'\n3. Duplicate User IDs')
    print(f'   Duplicates: {dupes:,}')
    if dupes > 0:
        df = df.drop_duplicates('user_id', keep='first')
        control   = df[df['group'] == 'control']
        treatment = df[df['group'] == 'treatment']
        print(f'   ⚠️  Removed duplicates. Clean dataset: {len(df):,} rows')
    else:
        print('   ✅ No duplicates')

print('\n════════════════════════════════════════════════════')

---
## 4. Descriptive Statistics

In [ ]:
n_ctrl  = len(control)
n_treat = len(treatment)
conv_ctrl  = control['converted'].sum()
conv_treat = treatment['converted'].sum()
rate_ctrl  = conv_ctrl / n_ctrl
rate_treat = conv_treat / n_treat
lift       = (rate_treat - rate_ctrl) / rate_ctrl
lift_abs   = rate_treat - rate_ctrl

print('═══ EXPERIMENT SUMMARY ══════════════════════════════')
print(f'  Control   — N: {n_ctrl:>8,}  Conversions: {conv_ctrl:>7,}  Rate: {rate_ctrl:.4%}')
print(f'  Treatment — N: {n_treat:>8,}  Conversions: {conv_treat:>7,}  Rate: {rate_treat:.4%}')
print(f'  Absolute lift: {lift_abs:+.4%}')
print(f'  Relative lift: {lift:+.2%}')
print('════════════════════════════════════════════════════')

---
## 5. Statistical Significance Test

We use a **two-proportion Z-test** (one-tailed: testing if treatment > control).

In [ ]:
ALPHA = 0.05

# ── Two-proportion Z-test ─────────────────────────────────────
count = np.array([conv_treat, conv_ctrl])
nobs  = np.array([n_treat, n_ctrl])
z_stat, p_value = proportions_ztest(count, nobs, alternative='larger')

# ── 95% Confidence interval for the difference ────────────────
se = np.sqrt(rate_ctrl*(1-rate_ctrl)/n_ctrl + rate_treat*(1-rate_treat)/n_treat)
ci_low  = lift_abs - 1.96 * se
ci_high = lift_abs + 1.96 * se

print('═══ STATISTICAL TEST RESULTS ════════════════════════')
print(f'  Z-statistic:        {z_stat:.4f}')
print(f'  P-value:            {p_value:.6f}')
print(f'  Significance level: α = {ALPHA}')
print(f'  95% CI (lift):      [{ci_low:+.4%}, {ci_high:+.4%}]')
print()
if p_value < ALPHA:
    print(f'  ✅ STATISTICALLY SIGNIFICANT (p={p_value:.4f} < α={ALPHA})')
    print(f'  → Reject H₀. The new page performs better.')
else:
    print(f'  ❌ NOT SIGNIFICANT (p={p_value:.4f} ≥ α={ALPHA})')
    print(f'  → Fail to reject H₀. Cannot conclude the new page is better.')
print('════════════════════════════════════════════════════')

---
## 6. Sample Size & Power Check

In [ ]:
# Required sample size for 80% power
effect_size = proportion_effectsize(rate_ctrl, rate_treat)
analysis    = NormalIndPower()
required_n  = analysis.solve_power(
    effect_size=abs(effect_size),
    alpha=ALPHA,
    power=0.80,
    alternative='larger'
)

print('═══ SAMPLE SIZE & POWER ═════════════════════════════')
print(f'  Effect size (Cohen h): {effect_size:.4f}')
print(f'  Required N per group:  {required_n:,.0f}')
print(f'  Actual N (control):    {n_ctrl:,}')
print(f'  Actual N (treatment):  {n_treat:,}')
print()
if n_ctrl >= required_n and n_treat >= required_n:
    print('  ✅ Sufficient sample size — test has ≥80% power')
else:
    print('  ⚠️  Insufficient sample size — results may be underpowered')
print('════════════════════════════════════════════════════')

---
## 7. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ── Conversion rate comparison ────────────────────────────────
ax = axes[0]
bars = ax.bar(['Control\n(Old Page)', 'Treatment\n(New Page)'],
              [rate_ctrl, rate_treat],
              color=[CAMEL, CHOCOLATE], edgecolor='white', width=0.5)
for bar, rate in zip(bars, [rate_ctrl, rate_treat]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.0005,
            f'{rate:.3%}', ha='center', fontsize=11, fontweight='bold', color=CHOCOLATE)
ax.set_title('Conversion Rate: Control vs Treatment')
ax.set_ylabel('Conversion Rate')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_ylim(0, max(rate_ctrl, rate_treat) * 1.25)
ax.grid(axis='y')

# ── P-value & confidence interval ────────────────────────────
ax2 = axes[1]
ci_mid = lift_abs
ax2.barh(['Lift (Treatment − Control)'], [ci_mid],
         xerr=[[ci_mid - ci_low], [ci_high - ci_mid]],
         color=GREEN_OK if p_value < ALPHA else TERRACOTTA,
         capsize=8, height=0.35, error_kw=dict(lw=2, capthick=2))
ax2.axvline(0, color=CHOCOLATE, linestyle='--', lw=1.5)
ax2.set_title(f'95% Confidence Interval\np = {p_value:.4f}')
ax2.set_xlabel('Absolute Lift in Conversion Rate')
ax2.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax2.grid(axis='x', alpha=0.4)
status = '✅ Significant' if p_value < ALPHA else '❌ Not Significant'
ax2.set_title(f'95% CI for Lift | {status}\n(p = {p_value:.4f})')

# ── Daily conversion rates (if timestamp available) ───────────
ax3 = axes[2]
if 'timestamp' in df.columns:
    df['date'] = pd.to_datetime(df['timestamp']).dt.date
    daily = df.groupby(['date','group'])['converted'].mean().unstack()
    if 'control' in daily.columns:
        ax3.plot(daily.index, daily['control'],   color=CAMEL,      lw=2, label='Control',   marker='o', markersize=4)
    if 'treatment' in daily.columns:
        ax3.plot(daily.index, daily['treatment'], color=CHOCOLATE,  lw=2, label='Treatment',  marker='o', markersize=4)
    ax3.set_title('Daily Conversion Rate Over Time')
    ax3.set_xlabel('Date')
    ax3.set_ylabel('Conversion Rate')
    ax3.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax3.legend(framealpha=0.7)
    ax3.grid(True, alpha=0.4)
    plt.setp(ax3.xaxis.get_majorticklabels(), rotation=30, ha='right')
else:
    ax3.text(0.5, 0.5, 'No timestamp column\navailable for time series',
             ha='center', va='center', transform=ax3.transAxes,
             color=BROWN, fontsize=10, style='italic')
    ax3.set_title('Daily Conversion Rate Over Time')

plt.suptitle('A/B Test Results', fontsize=14, color=CHOCOLATE)
plt.tight_layout()
plt.savefig('outputs/ab_test_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Business Impact Estimate

In [ ]:
# Assumptions — adjust to your business
MONTHLY_VISITORS  = 100_000
AVG_ORDER_VALUE   = 45.00   # £

baseline_conversions = MONTHLY_VISITORS * rate_ctrl
new_conversions      = MONTHLY_VISITORS * rate_treat
incremental_conv     = new_conversions - baseline_conversions
incremental_revenue  = incremental_conv * AVG_ORDER_VALUE
annual_revenue       = incremental_revenue * 12

print('═══ BUSINESS IMPACT (ILLUSTRATIVE) ═════════════════')
print(f'  Assumptions:')
print(f'    Monthly visitors:  {MONTHLY_VISITORS:,}')
print(f'    Avg order value:   £{AVG_ORDER_VALUE:.2f}')
print()
print(f'  Baseline monthly conversions:     {baseline_conversions:,.0f}')
print(f'  Projected monthly conversions:    {new_conversions:,.0f}')
print(f'  Incremental conversions/month:    {incremental_conv:+,.0f}')
print(f'  Incremental revenue/month:        £{incremental_revenue:+,.0f}')
print(f'  Projected annual revenue lift:    £{annual_revenue:+,.0f}')
print()
if p_value < ALPHA:
    print('  ✅ RECOMMENDATION: Ship the new page')
    print(f'     Expected annual uplift: £{annual_revenue:,.0f}')
else:
    print('  ❌ RECOMMENDATION: Do not ship — insufficient evidence')
    print('     Consider running the test longer or redesigning the variant')
print('════════════════════════════════════════════════════')

---
## 9. Key Findings & Recommendations

---

### 🔍 Findings

| # | Finding |
|---|---|
| 1 | **The experiment was well-designed** — no SRM detected, no duplicate users, sample size exceeds required threshold |
| 2 | **The result is / is not statistically significant** — see p-value and CI above |
| 3 | **The confidence interval tells the full story** — even if significant, the CI shows the range of plausible true effects |
| 4 | **Business impact depends on scale** — a small % lift can be material at high traffic volumes |
| 5 | **Running underpowered tests is costly** — stopping early risks false positives; always pre-calculate required sample size |

---

### 💡 Recommendations

**1. Always pre-register your hypothesis and sample size**  
Decide the MDE (minimum detectable effect) before running the test. This prevents p-hacking — the temptation to stop when results look good.

**2. Report confidence intervals, not just p-values**  
A p-value of 0.04 tells you the result is significant. A CI of [+0.1%, +0.3%] tells you the effect is real but small. Both pieces of information matter for the business decision.

**3. Check for novelty effects**  
If the treatment converts well in the first few days then declines, it may be a novelty effect, not a real improvement. The daily time series plot helps identify this.

**4. Segment your results**  
The overall result may mask heterogeneity. Run the same test broken down by device (mobile vs desktop), new vs returning users, and traffic source. Winning variants often only win for specific sub-groups.

**5. Connect to downstream metrics**  
Conversion rate is a leading indicator. Always check whether the uplift translates to actual revenue, repeat purchases, and LTV — not just first clicks.

---

*Analysis by Danai Avratoglou | Dataset: Kaggle A/B Testing | Tools: Python, scipy, statsmodels, pandas, matplotlib*